## Импорт библиотек

In [26]:
import pandas as pd
from pathlib import Path
import re
from datetime import datetime
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import rasterio
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from torchmetrics.classification import MulticlassF1Score
from glob import glob
import torch
import random
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
import os

## Residual Attention U-Net

In [27]:
def seed_everything(seed=0):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

seed_everything(0)

def conv3x3(in_planes, out_planes, stride=1):
    "3x3 convolution with padding"
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
           
        self.fc = nn.Sequential(nn.Conv2d(in_planes, in_planes // 16, 1, bias=False),
                               nn.ReLU(),
                               nn.Conv2d(in_planes // 16, in_planes, 1, bias=False))
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()

        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)

class ResidualBlock(nn.Module):
    def __init__(self, inputChannel, outputChannel, stride=1, downsample=None):
        super(ResidualBlock, self).__init__()
        self.conv1 = conv3x3(inputChannel, outputChannel, stride)
        self.bn1 = nn.BatchNorm2d(outputChannel)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(outputChannel, outputChannel)
        self.bn2 = nn.BatchNorm2d(outputChannel)
        self.downsample = downsample
        self.ca = ChannelAttention(outputChannel)
        self.sa = SpatialAttention()
        
    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        caOutput = self.ca(out)
        out = caOutput * out
        saOutput = self.sa(out)
        out = saOutput * out
        return out, saOutput

class BasicDownSample(nn.Module):
    def __init__(self, inputChannel, outputChannel):
        super().__init__()
        self.convolution = nn.Sequential(
            nn.Conv2d(inputChannel, outputChannel, kernel_size=3, padding=1),
            nn.BatchNorm2d(outputChannel),
            nn.LeakyReLU(0.2),
            nn.Conv2d(outputChannel, outputChannel, kernel_size=3, padding=1),
            nn.BatchNorm2d(outputChannel),
            nn.LeakyReLU(0.2),
            nn.AvgPool2d(2)
        )
    
    def forward(self,x):
        x = self.convolution(x)
        return x

class DownSampleWithAttention(nn.Module):
    def __init__(self, inputChannel, outputChannel):
        super().__init__()
        self.convolution = nn.Sequential(
            nn.Conv2d(inputChannel, outputChannel, kernel_size=3, padding=1),
            nn.BatchNorm2d(outputChannel),
            nn.LeakyReLU(0.2),
            nn.Conv2d(outputChannel, outputChannel, kernel_size=3, padding=1),
            nn.BatchNorm2d(outputChannel),
            nn.LeakyReLU(0.2),
            nn.AvgPool2d(2)
        )
        self.ca = ChannelAttention(outputChannel)
        self.sa = SpatialAttention()
    
    def forward(self,x):
        x = self.convolution(x)
        caOutput = self.ca(x)
        x = caOutput * x
        saOutput = self.sa(x)
        x = saOutput * x
        return x, saOutput

class BasicUpSample(nn.Module):
    def __init__(self, inputChannel, outputChannel):
        super().__init__()
        self.convolution = nn.Sequential(
            nn.Conv2d(inputChannel, outputChannel, kernel_size=3, padding=1),
            nn.BatchNorm2d(outputChannel),
            nn.LeakyReLU(0.2),
            nn.Conv2d(outputChannel, outputChannel, kernel_size=3, padding=1),
            nn.BatchNorm2d(outputChannel),
            nn.LeakyReLU(0.2)
        )
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
    
    def forward(self, x):
        x = self.upsample(x)
        x = self.convolution(x)
        return x
    
class UpSampleWithAttention(nn.Module):
    def __init__(self, inputChannel, outputChannel):
        super().__init__()
        self.convolution = nn.Sequential(
            nn.Conv2d(inputChannel, outputChannel, kernel_size=3, padding=1),
            nn.BatchNorm2d(outputChannel),
            nn.LeakyReLU(0.2),
            nn.Conv2d(outputChannel, outputChannel, kernel_size=3, padding=1),
            nn.BatchNorm2d(outputChannel),
            nn.LeakyReLU(0.2)
        )
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.ca = ChannelAttention(outputChannel)
        self.sa = SpatialAttention()
    
    def forward(self, x):
        x = self.upsample(x)
        x = self.convolution(x)
        caOutput = self.ca(x)
        x = caOutput * x
        saOutput = self.sa(x)
        x = saOutput * x
        return x, saOutput

class UNet(nn.Module):
  def __init__(self, inputChannel, outputChannel):
    super().__init__()
    self.downsample1 = BasicDownSample(inputChannel, 32)
    self.downsample2 = BasicDownSample(32, 64)
    self.downsample3 = BasicDownSample(64, 128)
    self.downsample4 = BasicDownSample(128, 256)
    self.downsample5 = BasicDownSample(256, 512)

    self.upsample1 = BasicUpSample(512, 256)
    self.upsample2 = BasicUpSample(512, 128)
    self.upsample3 = BasicUpSample(256, 64)
    self.upsample4 = BasicUpSample(128, 32)
    self.upsample5 = BasicUpSample(64, 32)
    self.classification = self.classification = nn.Sequential(
            nn.Conv2d(32, outputChannel, kernel_size=1),
        )

  def forward(self, x):
    scale128 = self.downsample1(x)
    scale64 = self.downsample2(scale128)
    scale32 = self.downsample3(scale64)
    scale16 = self.downsample4(scale32)
    scale8 = self.downsample5(scale16)
    upscale16 = self.upsample1(scale8)
    upscale16 = torch.cat([upscale16, scale16], dim=1)
    upscale32 = self.upsample2(upscale16)
    upscale32 = torch.cat([upscale32, scale32], dim=1)
    upscale64 = self.upsample3(upscale32)
    upscale64 = torch.cat([upscale64, scale64], dim=1)
    upscale128 = self.upsample4(upscale64)
    upscale128 = torch.cat([upscale128, scale128], dim=1)
    upscale256 = self.upsample5(upscale128)
    finaloutput = self.classification(upscale256)
    return finaloutput

class AttentionUNet(nn.Module):
  def __init__(self, inputChannel, outputChannel):
    super().__init__()
    self.downsample1 = DownSampleWithAttention(inputChannel, 32)
    self.downsample2 = DownSampleWithAttention(32, 64)
    self.downsample3 = DownSampleWithAttention(64, 128)
    self.downsample4 = DownSampleWithAttention(128, 256)
    self.downsample5 = DownSampleWithAttention(256, 512)

    self.upsample1 = UpSampleWithAttention(512, 256)
    self.upsample2 = UpSampleWithAttention(512, 128)
    self.upsample3 = UpSampleWithAttention(256, 64)
    self.upsample4 = UpSampleWithAttention(128, 32)
    self.upsample5 = UpSampleWithAttention(64, 32)
    self.classification = nn.Sequential(
            nn.Conv2d(32, outputChannel, kernel_size=1),
        )

  def forward(self, x):
    scale128, sa128down = self.downsample1(x)
    scale64, sa64down = self.downsample2(scale128)
    scale32, sa32down = self.downsample3(scale64)
    scale16, sa64down = self.downsample4(scale32)
    scale8, sa8down = self.downsample5(scale16)
    upscale16, sa16up = self.upsample1(scale8)
    upscale16 = torch.cat([upscale16, scale16], dim=1)
    upscale32, sa32up = self.upsample2(upscale16)
    upscale32 = torch.cat([upscale32, scale32], dim=1)
    upscale64, sa64up = self.upsample3(upscale32)
    upscale64 = torch.cat([upscale64, scale64], dim=1)
    upscale128, sa128up = self.upsample4(upscale64)
    upscale128 = torch.cat([upscale128, scale128], dim=1)
    upscale256, sa256up = self.upsample5(upscale128)
    finaloutput = self.classification(upscale256)
    return finaloutput

class ResidualAttentionUNet(nn.Module):
  def __init__(self, inputChannel, outputChannel):
    super().__init__()
    self.downsample1 = DownSampleWithAttention(inputChannel, 32)
    self.downsample2 = DownSampleWithAttention(32, 64)
    self.downsample3 = DownSampleWithAttention(64, 128)
    self.downsample4 = DownSampleWithAttention(128, 256)
    self.downsample5 = DownSampleWithAttention(256, 512)

    self.residualBlock1 = ResidualBlock(512, 512)
    self.residualBlock2 = ResidualBlock(512, 512)
    self.residualBlock3 = ResidualBlock(512, 512)

    self.upsample1 = UpSampleWithAttention(512, 256)
    self.upsample2 = UpSampleWithAttention(512, 128)
    self.upsample3 = UpSampleWithAttention(256, 64)
    self.upsample4 = UpSampleWithAttention(128, 32)
    self.upsample5 = UpSampleWithAttention(64, 32)
    self.classification = nn.Sequential(
            nn.Conv2d(32, outputChannel, kernel_size=1),
        )

  def forward(self, x):
    scale128, sa128down = self.downsample1(x)
    scale64, sa64down = self.downsample2(scale128)
    scale32, sa32down = self.downsample3(scale64)
    scale16, sa64down = self.downsample4(scale32)
    scale8, sa8down = self.downsample5(scale16)
    scale8, sa8down = self.residualBlock1(scale8)
    scale8, sa8down = self.residualBlock2(scale8)
    scale8, sa8down = self.residualBlock3(scale8)
    upscale16, sa16up = self.upsample1(scale8)
    upscale16 = torch.cat([upscale16, scale16], dim=1)
    upscale32, sa32up = self.upsample2(upscale16)
    upscale32 = torch.cat([upscale32, scale32], dim=1)
    upscale64, sa64up = self.upsample3(upscale32)
    upscale64 = torch.cat([upscale64, scale64], dim=1)
    upscale128, sa128up = self.upsample4(upscale64)
    upscale128 = torch.cat([upscale128, scale128], dim=1)
    upscale256, sa256up = self.upsample5(upscale128)
    finaloutput = self.classification(upscale256)
    return finaloutput

## Dataset

In [28]:
import os
from pathlib import Path

import numpy as np
import rasterio
import torch
from torch.utils.data import Dataset

CLASS_NAMES = {
    1: "Marine Debris",
    2: "Dense Sargassum",
    3: "Sparse Sargassum",
    4: "Natural Organic Material",
    5: "Ship",
    6: "Clouds",
    7: "Marine Water",
    8: "Sediment-Laden Water",
    9: "Foam",
    10: "Turbid Water",
    11: "Shallow Water",
    12: "Waves",
    13: "Cloud Shadows",
    14: "Wakes",
    15: "Mixed Water",
}
DEBRIS_CLASS_DN = 1  # положительный класс для бинарной задачи
IGNORE_INDEX = 255   # пиксели с низкой уверенностью / вне разметки

# Индексы каналов Sentinel-2 в порядке, как они лежат в MARIDA-патчах
BAND_NAMES = ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
BAND_IDX = {name: i for i, name in enumerate(BAND_NAMES)}

REFLECTANCE_SCALE = 1  # нормализация отражательной способности S2 L1C

def augment_pair( image, mask ):
    # image: [C, H, W], mask: [H, W]
    if random.random() < 0.5:
        image = torch.flip( image, dims=(-1,) )
        mask = torch.flip( mask, dims=(-1,) )

    if random.random() < 0.5:
        image = torch.flip( image, dims=(-2,) )
        mask = torch.flip( mask, dims=(-2,) )

    k = random.randrange( 4 )
    image = torch.rot90( image, k, dims=(-2, -1) )
    mask = torch.rot90( mask, k, dims=(-2, -1) )

    return image, mask

def compute_fdi(bands):
    nir = bands[BAND_IDX["B8"]].astype(np.float32) / REFLECTANCE_SCALE
    red_edge = bands[BAND_IDX["B6"]].astype(np.float32) / REFLECTANCE_SCALE
    swir = bands[BAND_IDX["B11"]].astype(np.float32) / REFLECTANCE_SCALE

    lambda_nir, lambda_re, lambda_swir = 833.0, 740.0, 1610.0
    ratio = (lambda_nir - lambda_re) / (lambda_swir - lambda_re)
    nir_baseline = red_edge + (swir - red_edge) * ratio
    return nir - nir_baseline

def compute_fai(bands):
    nir = bands[BAND_IDX["B8A"]].astype(np.float32) / REFLECTANCE_SCALE
    red = bands[BAND_IDX["B4"]].astype(np.float32) / REFLECTANCE_SCALE
    swir = bands[BAND_IDX["B11"]].astype(np.float32) / REFLECTANCE_SCALE

    lambda_nir, lambda_red, lambda_swir = 865.0, 665.0, 1610.0
    ratio = (lambda_nir - lambda_red) / (lambda_swir - lambda_red)
    red_baseline = red + (swir - red) * ratio
    return nir - red_baseline


def compute_ndvi(bands, eps=1e-3):
    nir = bands[BAND_IDX["B8"]].astype(np.float32)
    red = bands[BAND_IDX["B4"]].astype(np.float32)
    ndvi = (nir - red) / (nir + red + eps)
    return np.clip(ndvi, -1.0, 1.0)

def compute_ndwi(bands, eps=1e-4):
    green = bands[BAND_IDX['B3']].astype(np.float32)
    nir = bands[BAND_IDX['B8']].astype(np.float32)
    return np.clip((green - nir) / (green + nir + eps), -1.0, 1.0)

class MaridaBinaryDataset(Dataset):
    def __init__(
        self,
        root_dir,
        split="train",
        add_indices=True,
        min_confidence=None,
        transform=None,
        band_stats=None,
        return_multiclass=False,
        augment=False,
    ):
        """
        Args:
            root_dir: путь до распакованной MARIDA (содержит patches/ и splits/).
            split: "train", "val" или "test".
            add_indices: добавлять ли FDI/NDVI/FAI как доп. каналы к спектру.
            min_confidence: если задано (1, 2 или 3), пиксели с confidence
                            хуже порога помечаются IGNORE_INDEX и исключаются
                            из лосса (CrossEntropyLoss/BCE с маскированием).
            transform: albumentations-совместимая аугментация.
            band_stats: dict {"mean": [...], "std": [...]} для z-score
                        нормализации поверх базового /10000 масштабирования.
            return_multiclass: если True, __getitem__ дополнительно возвращает
                        оригинальную мультиклассовую маску (1..15) — нужно для
                        разбора ложных срабатываний по confounder-классам
                        (Sargassum, Foam, Ships) при анализе ошибок (критерий О3).
        """
        self.root_dir = Path(root_dir)
        self.patches_dir = self.root_dir / "patches"
        self.add_indices = add_indices
        self.min_confidence = min_confidence
        self.band_stats = band_stats
        self.return_multiclass = return_multiclass
        self.augment = augment

        split_file = self.root_dir / "splits" / f"{split}_X.txt"
        if not split_file.exists():
            raise FileNotFoundError(
                f"Split file not found: {split_file}. "
                "Используйте готовые сплиты из "
                "https://github.com/marine-debris/marine-debris.github.io"
            )
        with open(split_file) as f:
            self.patch_ids = [line.strip() for line in f if line.strip()]

    def __len__(self):
        return len(self.patch_ids)

    def _paths_for(self, patch_id):
        # patch_id из train_X.txt: "1-12-19_48MYU_0" (без сенсора)
        # реальные папки/файлы на диске: "S2_1-12-19_48MYU_0.tif"
        full_id = patch_id if patch_id.startswith("S2_") else f"S2_{patch_id}"
    
        tile_dir = "_".join(full_id.split("_")[:-1])  # "S2_1-12-19_48MYU"
        base = self.patches_dir / tile_dir / full_id
    
        return (
            base.with_name(base.name + ".tif"),
            base.with_name(base.name + "_cl.tif"),
            base.with_name(base.name + "_conf.tif"),
        )

    def __getitem__(self, idx):
        patch_id = self.patch_ids[idx]
        img_path, cl_path, conf_path = self._paths_for(patch_id)

        with rasterio.open(img_path) as src:
            raw_bands = src.read().astype(np.float32)  # (11, H, W)

        with rasterio.open(cl_path) as src:
            multiclass_mask = src.read(1).astype(np.int64)  # DN 1..15

        conf = None
        if conf_path.exists():
            with rasterio.open(conf_path) as src:
                conf = src.read(1).astype(np.int64)  # 1=High,2=Mod,3=Low
        # Если хотя бы одна исходная полоса нечисловая, не обучаемся на этом пикселе.
        invalid_pixel = ~np.isfinite(raw_bands).all(axis=0)

        # Замена нужна до вычисления FDI/NDVI/FAI.
        raw_bands = np.nan_to_num(
            raw_bands, nan=0.0, posinf=0.0, neginf=0.0
        )
        # --- бинаризация: 1 = Marine Debris (мусор), 0 = всё остальное ---
        binary_label = np.full(
            multiclass_mask.shape, IGNORE_INDEX, dtype=np.int64
        )
        binary_label[(multiclass_mask >= 2) & (multiclass_mask <= 15)] = 0
        binary_label[multiclass_mask == DEBRIS_CLASS_DN] = 1
        binary_label[invalid_pixel] = IGNORE_INDEX

        # --- фильтрация по уверенности (до аугментаций) ---
        if self.min_confidence is not None and conf is not None:
            low_conf = conf > self.min_confidence
            binary_label[low_conf] = IGNORE_INDEX

        # --- спектральные индексы на сырых (ненормализованных) значениях ---
        extra_channels = []
        if self.add_indices:
            fdi = compute_fdi(raw_bands)
            ndvi = compute_ndvi(raw_bands)
            fai = compute_fai(raw_bands)
            ndwi = compute_ndwi(raw_bands)
            extra_channels = [fdi, ndvi, fai, ndwi]

        # --- нормализация спектральных каналов ---
        bands = raw_bands / REFLECTANCE_SCALE
        if self.band_stats is not None:
            mean = np.array(self.band_stats["mean"], dtype=np.float32).reshape(-1, 1, 1)
            std = np.array(self.band_stats["std"], dtype=np.float32).reshape(-1, 1, 1)
            bands = (bands - mean) / (std + 1e-6)

        if extra_channels:
            extra = np.stack(extra_channels, axis=0)
            image = np.concatenate([bands, extra], axis=0)  # (11+3, H, W)
        else:
            image = bands

        image_t = torch.from_numpy(np.ascontiguousarray(image)).float()
        label_t = torch.from_numpy(np.ascontiguousarray(binary_label)).long()

        if self.augment:
            image_t, label_t = augment_pair(image_t, label_t)

        if self.return_multiclass:
            multiclass_t = torch.from_numpy(np.ascontiguousarray(multiclass_mask)).long()
            return image_t, label_t, multiclass_t

        return image_t, label_t

    @property
    def num_input_channels(self):
        return len(BAND_NAMES) + (3 if self.add_indices else 0)


def compute_band_statistics(dataset, max_samples=200):
    """Считает mean/std по каналам на подвыборке — для band_stats."""
    sums, sq_sums, count = None, None, 0
    n = min(max_samples, len(dataset))
    for i in range(n):
        img, _ = dataset[i][:2]
        img = img.numpy()
        if sums is None:
            sums = np.zeros(img.shape[0], dtype=np.float64)
            sq_sums = np.zeros(img.shape[0], dtype=np.float64)
        sums += img.reshape(img.shape[0], -1).sum(axis=1)
        sq_sums += (img.reshape(img.shape[0], -1) ** 2).sum(axis=1)
        count += img.shape[1] * img.shape[2]

    mean = sums / count
    std = np.sqrt(sq_sums / count - mean ** 2)
    return {"mean": mean.tolist(), "std": std.tolist()}


def compute_class_balance(dataset, max_samples=None):
    """
    Считает долю фон/мусор/игнор пикселей по датасету — нужно, чтобы
    подобрать pos_weight для BCEWithLogitsLoss.
    """
    n_bg, n_debris, n_ignore = 0, 0, 0
    n = len(dataset) if max_samples is None else min(max_samples, len(dataset))
    for i in range(n):
        _, label = dataset[i][:2]
        n_bg += (label == 0).sum().item()
        n_debris += (label == 1).sum().item()
        n_ignore += (label == IGNORE_INDEX).sum().item()
    return {"background": n_bg, "debris": n_debris, "ignored": n_ignore,
            "pos_weight_suggestion": n_bg / max(n_debris, 1)}


def analyze_false_positives(binary_pred, binary_label, multiclass_mask):
    """
    Разбор ложных срабатываний по confounder-классам MARIDA — нужно для
    критерия О3 (различение пластика и сложного морского фона).

    Args:
        binary_pred, binary_label, multiclass_mask: тензоры/массивы одинаковой
            формы (H, W) или (B, H, W).
    Returns:
        dict {class_name: fp_pixel_count}
    """
    binary_pred = np.asarray(binary_pred)
    binary_label = np.asarray(binary_label)
    multiclass_mask = np.asarray(multiclass_mask)

    fp_mask = (binary_pred == 1) & (binary_label == 0)
    result = {}
    for dn, name in CLASS_NAMES.items():
        if dn == DEBRIS_CLASS_DN:
            continue
        result[name] = int((fp_mask & (multiclass_mask == dn)).sum())
    return result

In [29]:
import math
import torch
device = 'cuda'
train_ds = MaridaBinaryDataset("MARIDA", split="train", add_indices=True, augment=True, min_confidence=None)
balance = compute_class_balance(train_ds, max_samples=100)
ratio = balance["background"] / max(balance["debris"], 1)
pos_weight = torch.tensor([balance["pos_weight_suggestion"]])
pos_weight = torch.tensor(
    [math.log1p(ratio)], dtype=torch.float32, device=device
)

val_ds = MaridaBinaryDataset("MARIDA", split="val", add_indices=True, min_confidence=None, return_multiclass=True)
test_ds = MaridaBinaryDataset("MARIDA", split="test", add_indices=True, min_confidence=None, return_multiclass=True)

## Hard negative mining

In [30]:
from torch.utils.data import WeightedRandomSampler

sampler_ds = MaridaBinaryDataset(
    "MARIDA",
    split="train",
    add_indices=False,
    min_confidence=None,
    return_multiclass=True,
    augment=False,
)

weights = []
for i in range(len(sampler_ds)):
    _, label, multiclass = sampler_ds[i]

    valid_bg = label == 0
    has_debris = bool((label == 1).any())
    has_ship = bool(((multiclass == 5) & valid_bg).any())
    has_sargassum = bool(
        (((multiclass == 2) | (multiclass == 3)) & valid_bg).any()
    )
    has_wakes = bool(((multiclass == 14) & valid_bg).any())
    has_foam = bool(((multiclass == 9) & valid_bg).any())
    has_sedimen_water = bool(((multiclass == 8) & valid_bg).any())
    # Разделяем веса: патчи с кораблями БЕЗ мусора должны быть приоритетными hard negatives
    if has_debris and has_ship:
        w = 12.0
    elif has_ship and not has_debris:
        # Фокусируем модель на подавлении судов на чистой воде
        w = 15.0  
    elif has_debris:
        w = 9.0
    elif has_wakes:
        w = 6.0
    elif has_sargassum or has_foam:
        w = 4.0
    else:
        w = 1.0

    weights.append(w)

sampler = WeightedRandomSampler(
    weights,
    num_samples=len(weights),
    replacement=True,
)

## Binary OHEM loss

In [31]:
class BinaryOHEMLoss(nn.Module):
    def __init__(self, ratio=0.1, pos_weight=None):
        super(BinaryOHEMLoss, self).__init__()
        self.ratio = ratio
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        loss = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction='none'
        )
        loss_flat = loss.view(-1)
        num_samples = loss_flat.numel()
        num_hard = max(int(num_samples * self.ratio), 1)  # защита от num_hard=0 на маленьких батчах
        topk_losses, _ = torch.topk(loss_flat, k=num_hard, sorted=False)
        return topk_losses.mean()

class ShipPenalizedOHEMLoss(nn.Module):
    def __init__(self, ratio=0.3, pos_weight=None, ship_penalty=4.0):
        super().__init__()
        self.ratio = ratio
        self.pos_weight = pos_weight
        self.ship_penalty = ship_penalty

    def forward(self, logits, targets, multiclass_mask=None):
        loss = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction='none'
        )
        
        # Усиливаем лосс на пикселях судов (DN 5), если модель предсказывает там мусор
        if multiclass_mask is not None:
            ship_pixels = (multiclass_mask == 5) & (targets == 0)
            loss = torch.where(ship_pixels, loss * self.ship_penalty, loss)

        loss_flat = loss.view(-1)
        num_hard = max(int(loss_flat.numel() * self.ratio), 1)
        topk_losses, _ = torch.topk(loss_flat, k=num_hard, sorted=False)
        return topk_losses.mean()

## Loaders

In [32]:
device = 'cuda'
model = ResidualAttentionUNet(inputChannel=15, outputChannel=1).to(device, memory_format=torch.channels_last)

train_loader = DataLoader(train_ds, batch_size=24, sampler=sampler, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=24, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=24, shuffle=False, num_workers=0, pin_memory=True)

In [10]:
from torch.optim.lr_scheduler import StepLR
from torch.optim.lr_scheduler import CosineAnnealingLR

NUM_EPOCHS = 50
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))


IGNORE_INDEX = 255
criterion = ShipPenalizedOHEMLoss(
    ratio=0.30,
    # pos_weight=pos_weight.to(device),
)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
device = 'cuda'
NUM_CLASSES = 2

In [21]:
pos_weight = pos_weight.to(device)

from torchmetrics.classification import BinaryF1Score
f1_metric = BinaryF1Score(threshold=0.5, ignore_index=IGNORE_INDEX).to(device)


best_f1 = -10000
NUM_EPOCHS = 50

for epoch in tqdm(range(NUM_EPOCHS)):
    model.train()
    train_loss = 0.0

    for images, masks in train_loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            outputs = model(images).squeeze(1)
            valid_mask = (masks != IGNORE_INDEX)
            outputs_valid = outputs[valid_mask]
            targets_valid = masks[valid_mask].float()

            loss = criterion(outputs_valid, targets_valid)  # OHEM вместо обычного BCE

        if amp_dtype == torch.bfloat16:
            loss.backward()
            optimizer.step()
        else:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        train_loss += loss.detach().item()

    model.eval()
    f1_metric.reset()
    val_loss = 0.0

    with torch.no_grad():
        for images, masks, _ in val_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                outputs = model(images).squeeze(1)
                valid_mask = (masks != IGNORE_INDEX)
                outputs_valid = outputs[valid_mask]
                targets_valid = masks[valid_mask].float()

                # для val используем обычный BCE, а не OHEM — чтобы val_loss был сравним между эпохами
                loss = F.binary_cross_entropy_with_logits(
                    outputs_valid, targets_valid, pos_weight=pos_weight
                )

            val_loss += loss.item()
            probs = torch.sigmoid(outputs)
            f1_metric.update(probs, masks)

    f1_epoch = f1_metric.compute().item()
    val_loss /= len(val_loader)
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]
    print(
        f"\nEpoch {epoch + 1}/{NUM_EPOCHS} |"
        f"LR: {current_lr:.7f}\n"
        f"Train Loss: {train_loss / len(train_loader):.4f} | "
        f"Val Loss: {val_loss:.4f} | Val F1: {f1_epoch:.4f}\n"
    )
    if f1_epoch > best_f1:
        best_f1 = f1_epoch
        torch.save(model.state_dict(), "resattunet_run9_ship-OHEM010_weighted-sampler_none-confidence_new-weights.pt")
        print("f1 checkpoint, model saved")

  2%|▏         | 1/50 [00:55<45:14, 55.39s/it]


Epoch 1/50 |LR: 0.0009990
Train Loss: 0.6762 | Val Loss: 0.6837 | Val F1: 0.0000

f1 checkpoint, model saved


  4%|▍         | 2/50 [01:53<45:28, 56.85s/it]


Epoch 2/50 |LR: 0.0009961
Train Loss: 0.6259 | Val Loss: 0.5721 | Val F1: 0.0131

f1 checkpoint, model saved


  6%|▌         | 3/50 [02:46<43:19, 55.32s/it]

...


Epoch 48/50 |LR: 0.0000049
Train Loss: 0.1051 | Val Loss: 0.0937 | Val F1: 0.7965



 98%|█████████▊| 49/50 [43:35<00:53, 53.10s/it]


Epoch 49/50 |LR: 0.0000020
Train Loss: 0.1049 | Val Loss: 0.0923 | Val F1: 0.7754



100%|██████████| 50/50 [44:28<00:00, 53.38s/it]


Epoch 50/50 |LR: 0.0000010
Train Loss: 0.1227 | Val Loss: 0.0788 | Val F1: 0.8150

f1 checkpoint, model saved


In [15]:
import numpy as np
from torchmetrics.classification import BinaryF1Score

model.load_state_dict(torch.load("resattunet_run9_ship-OHEM010_weighted-sampler_none-confidence_new-weights.pt"))
model.eval()

thresholds = np.arange(0.45, 0.60, 0.01)
best_threshold, best_f1 = 0.5, 0.0

model.eval()
with torch.no_grad():
    all_probs, all_masks = [], []
    for images, masks, _ in val_loader:
        images = images.to(device)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            outputs = model(images).squeeze(1)
        probs = torch.sigmoid(outputs).cpu()
        all_probs.append(probs)
        all_masks.append(masks)

    all_probs = torch.cat(all_probs)
    all_masks = torch.cat(all_masks)

for t in thresholds:
    f1_metric = BinaryF1Score(threshold=float(t), ignore_index=IGNORE_INDEX)
    f1 = f1_metric(all_probs, all_masks).item()
    print(f"threshold={t:.2f}  F1={f1:.4f}")
    if f1 > best_f1:
        best_f1, best_threshold = f1, t

print(f"\nЛучший порог: {best_threshold:.2f}, F1={best_f1:.4f}")

threshold=0.45  F1=0.8136
threshold=0.46  F1=0.8147
threshold=0.47  F1=0.8148
threshold=0.48  F1=0.8152
threshold=0.49  F1=0.8153
threshold=0.50  F1=0.8150
threshold=0.51  F1=0.8150
threshold=0.52  F1=0.8125
threshold=0.53  F1=0.8106
threshold=0.54  F1=0.8104
threshold=0.55  F1=0.8079
threshold=0.56  F1=0.8051
threshold=0.57  F1=0.8051
threshold=0.58  F1=0.8045
threshold=0.59  F1=0.8025

Лучший порог: 0.49, F1=0.8153


In [16]:
from torchmetrics.classification import BinaryPrecision, BinaryRecall, BinaryJaccardIndex

precision_metric = BinaryPrecision(threshold=0.49, ignore_index=IGNORE_INDEX).to(device)
recall_metric = BinaryRecall(threshold=0.49, ignore_index=IGNORE_INDEX).to(device)
iou_metric = BinaryJaccardIndex(threshold=0.49, ignore_index=IGNORE_INDEX).to(device)

with torch.no_grad():
    for images, masks, _ in test_loader:
        images, masks = images.to(device), masks.to(device)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            outputs = model(images).squeeze(1)
        probs = torch.sigmoid(outputs)
        precision_metric.update(probs, masks)
        recall_metric.update(probs, masks)
        iou_metric.update(probs, masks)

print(f"Precision: {precision_metric.compute().item():.4f}")
print(f"Recall: {recall_metric.compute().item():.4f}")
print(f"IoU: {iou_metric.compute().item():.4f}")

Precision: 0.6905
Recall: 0.6850
IoU: 0.5241


In [17]:
model.eval()
fp_totals = {}

with torch.no_grad():
    for images, masks, multiclass in test_loader:
        images = images.to(device)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            outputs = model(images).squeeze(1)
        preds = (torch.sigmoid(outputs) > 0.5).long().cpu()

        for i in range(preds.shape[0]):
            fp_by_class = analyze_false_positives(
                preds[i].numpy(), masks[i].numpy(), multiclass[i].numpy()
            )
            for k, v in fp_by_class.items():
                fp_totals[k] = fp_totals.get(k, 0) + v

for k, v in sorted(fp_totals.items(), key=lambda x: -x[1]):
    print(f"{k}: {v} ложных пикселей")

Ship: 37 ложных пикселей
Natural Organic Material: 35 ложных пикселей
Mixed Water: 18 ложных пикселей
Marine Water: 8 ложных пикселей
Waves: 7 ложных пикселей
Sparse Sargassum: 4 ложных пикселей
Clouds: 2 ложных пикселей
Dense Sargassum: 0 ложных пикселей
Sediment-Laden Water: 0 ложных пикселей
Foam: 0 ложных пикселей
Turbid Water: 0 ложных пикселей
Shallow Water: 0 ложных пикселей
Cloud Shadows: 0 ложных пикселей
Wakes: 0 ложных пикселей


In [18]:
from sklearn.metrics import average_precision_score
import numpy as np
import torch

def evaluate_ap(model, loader):
    y_true_parts = []
    y_score_parts = []

    model.eval()
    with torch.inference_mode():
        for images, masks, _ in loader:
            images = images.to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                logits = model(images).squeeze(1)

            probs = torch.sigmoid(logits.float()).cpu()
            valid = masks != IGNORE_INDEX

            y_true_parts.append(masks[valid].numpy().astype(np.uint8))
            y_score_parts.append(probs[valid].numpy())

    y_true = np.concatenate(y_true_parts)
    y_score = np.concatenate(y_score_parts)

    if not np.any(y_true == 1):
        raise ValueError("В выборке нет валидных положительных пикселей")

    ap = average_precision_score(y_true, y_score)
    prevalence = y_true.mean()

    return ap, prevalence, len(y_true)

for name, loader in [("val", val_loader), ("test", test_loader)]:
    ap, prevalence, n = evaluate_ap(model, loader)
    print(
        f"{name}: AP={ap:.4f}, "
        f"доля мусора={prevalence:.4%}, "
        f"валидных пикселей={n}"
    )

val: AP=0.8195, доля мусора=0.5045%, валидных пикселей=213102
test: AP=0.6506, доля мусора=0.1955%, валидных пикселей=194863


In [11]:
model = ResidualAttentionUNet(inputChannel=15, outputChannel=1).to(device, memory_format=torch.channels_last)
model.load_state_dict(torch.load("resattunet_run9_ship-OHEM010_weighted-sampler_none-confidence_new-weights.pt"))
model.eval()

ResidualAttentionUNet(
  (downsample1): DownSampleWithAttention(
    (convolution): Sequential(
      (0): Conv2d(15, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.2)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): LeakyReLU(negative_slope=0.2)
      (6): AvgPool2d(kernel_size=2, stride=2, padding=0)
    )
    (ca): ChannelAttention(
      (avg_pool): AdaptiveAvgPool2d(output_size=1)
      (max_pool): AdaptiveMaxPool2d(output_size=1)
      (fc): Sequential(
        (0): Conv2d(32, 2, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): ReLU()
        (2): Conv2d(2, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
      )
      (sigmoid): Sigmoid()
    )
    (sa): SpatialAttention(
      (conv1): Conv2d(2, 1, kern

In [24]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score

IGNORE_INDEX = 255
FEATURE_NAMES = [
    "B1", "B2", "B3", "B4", "B5", "B6", "B7",
    "B8", "B8A", "B11", "B12",
    "FDI", "NDVI", "FAI", "NDWI",
]
XGB_CHANNELS = list( range( 15 ) )


def pixels_for_xgboost( dataset, max_bg_per_patch=None, seed=0 ):
    rng = np.random.default_rng( seed )
    x_parts, y_parts = [], []

    for i in range( len( dataset ) ):
        image, mask = dataset[i][:2]

        image = image.numpy()[:15]
        mask = mask.numpy()

        x = image.transpose( 1, 2, 0 ).reshape( -1, 15 )
        y = mask.reshape( -1 )

        valid = ( y != IGNORE_INDEX )
        positive_idx = np.flatnonzero( valid & ( y == 1 ) )
        background_idx = np.flatnonzero( valid & ( y == 0 ) )

        if max_bg_per_patch is not None and len( background_idx ) > max_bg_per_patch:
            background_idx = rng.choice(
                background_idx,
                size=max_bg_per_patch,
                replace=False,
            )

        selected = np.concatenate( [positive_idx, background_idx] )
        x_parts.append( x[selected] )
        y_parts.append( y[selected] )

    return (
        np.concatenate( x_parts ).astype( np.float32 ),
        np.concatenate( y_parts ).astype( np.uint8 ),
    )


# Отдельный train-датасет БЕЗ аугментаций: деревьям флипы не нужны.
xgb_train_ds = MaridaBinaryDataset(
    "MARIDA",
    split="train",
    add_indices=True,
    augment=False,
    min_confidence=None,
)

# Проверь, что текущая версия Dataset действительно возвращает 15 каналов.
assert xgb_train_ds[0][0].shape[0] == 15, (
    "Dataset сейчас возвращает не 15 каналов. "
    "Верни версию без PI/WRI или явно зафиксируй первые 15 каналов "
    "в том же порядке, что и у baseline."
)

X_train, y_train = pixels_for_xgboost(
    xgb_train_ds,
    max_bg_per_patch=300,
    seed=0,
)

# Val оставляем БЕЗ undersampling: он должен отражать реальное распределение.
X_val, y_val = pixels_for_xgboost(
    val_ds,
    max_bg_per_patch=None,
)

print( "Train:", X_train.shape, np.bincount( y_train ) )
print( "Val:  ", X_val.shape, np.bincount( y_val ) )

# Вес рассчитываем ПОСЛЕ отбора train-пикселей.
neg = np.count_nonzero( y_train == 0 )
pos = np.count_nonzero( y_train == 1 )
assert pos > 0

# Для первого прогона не используем одновременно полное отношение neg/pos
# и сильный undersampling: начнём с умеренного веса.
scale_pos_weight = min( neg / pos, 10.0 )
print( "scale_pos_weight:", scale_pos_weight )

xgb_model = XGBClassifier(
    n_estimators=2000,
    max_depth=20,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.9,
    min_child_weight=3,
    objective="binary:logistic",
    tree_method="hist",
    device="cuda",
    eval_metric="aucpr",
    early_stopping_rounds=40,
    scale_pos_weight=scale_pos_weight,
    random_state=0,
    n_jobs=-1,
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[( X_val, y_val )],
    verbose=False,
)

val_proba = xgb_model.predict_proba( X_val )[:, 1]

print( "Best iteration:", xgb_model.best_iteration )
print( "Val AP:", average_precision_score( y_val, val_proba ) )

Train: (82197, 15) [80254  1943]
Val:   (213102, 15) [212027   1075]
scale_pos_weight: 10.0
Best iteration: 396
Val AP: 0.8659403785068138


In [25]:
import numpy as np
import torch

from sklearn.metrics import (
    average_precision_score,
    f1_score,
    jaccard_score,
    precision_score,
    recall_score,
)

# Порядок в твоём датасете:
# B1, B2, B3, B4, B5, B6, B7, B8, B8A, B11, B12,
# FDI, NDVI, FAI, NDWI
XGB_CHANNELS = list(range(15))  # ИЗМЕНИ, если XGBoost обучен на других признаках
IGNORE_INDEX = 255


def collect_ensemble_inputs( loader, model, xgb_model ):
    unet_parts, xgb_parts, label_parts = [], [], []

    model.eval()

    with torch.inference_mode():
        for images, masks, _ in loader:
            # XGBoost: каждый пиксель становится отдельной строкой.
            x = images[:, XGB_CHANNELS].permute( 0, 2, 3, 1 )
            x = x.reshape( -1, len( XGB_CHANNELS ) ).numpy()
            xgb_prob = xgb_model.predict_proba( x )[:, 1]

            images_gpu = images.to( device, non_blocking=True )
            with torch.autocast( device_type="cuda", dtype=amp_dtype ):
                logits = model( images_gpu ).squeeze( 1 )

            unet_prob = torch.sigmoid( logits.float() ).cpu().numpy()

            valid = masks.numpy() != IGNORE_INDEX
            unet_parts.append( unet_prob[valid] )
            xgb_parts.append( xgb_prob.reshape( masks.shape )[valid] )
            label_parts.append( masks.numpy()[valid] )

    return (
        np.concatenate( unet_parts ),
        np.concatenate( xgb_parts ),
        np.concatenate( label_parts ).astype( np.uint8 ),
    )


# ВАЖНО: здесь model уже должен содержать веса лучшего baseline,
# а не веса неудачного run с PI/WRI.
unet_val, xgb_val, y_val = collect_ensemble_inputs(
    val_loader, model, xgb_model
)

best = None

for w_xgb in np.arange( 0.0, 1.01, 0.1 ):
    combined = ( w_xgb * xgb_val + ( 1.0 - w_xgb ) * unet_val )

    for threshold in np.arange( 0.05, 1.0, 0.01 ):
        pred = combined >= threshold
        score = f1_score( y_val, pred, zero_division=0 )

        if best is None or score > best["val_f1"]:
            best = {
                "w_xgb": float( w_xgb ),
                "threshold": float( threshold ),
                "val_f1": float( score ),
            }

print( "Лучшая конфигурация на val:", best )

unet_test, xgb_test, y_test = collect_ensemble_inputs(test_loader, model, xgb_model)

combined_test = (best["w_xgb"] * xgb_test + ( 1.0 - best["w_xgb"] ) * unet_test)
pred_test = combined_test >= best["threshold"]

print( "Test Precision:", precision_score(y_test, pred_test, zero_division=0))
print( "Test Recall:   ", recall_score(y_test, pred_test, zero_division=0))
print( "Test F1:       ", f1_score(y_test, pred_test, zero_division=0))
print( "Test IoU:      ", jaccard_score(y_test, pred_test, zero_division=0))
print( "Test AP:       ", average_precision_score(y_test, combined_test))

Лучшая конфигурация на val: {'w_xgb': 0.5, 'threshold': 0.5000000000000001, 'val_f1': 0.9071090047393365}
Test Precision: 0.8229665071770335
Test Recall:    0.9028871391076115
Test F1:        0.8610763454317898
Test IoU:       0.756043956043956
Test AP:        0.9051431245522558
